In [1]:
import os
print(os.getcwd())
os.chdir('/home/fatemeh/thesis/kinodata-3D-affinity-prediction')
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/prob
/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [2]:
import kinodata.configuration as cfg

from prob.utils import get_model_dir, get_model_ckpt, get_gnn_config_path, get_split_file, get_out_dir
from prob.utils import build_kd_ds, build_gnn_model, load_config
from prob.prob_ds_helpers import run_fold

from torch_geometric.loader import DataLoader
import torch
from tqdm import tqdm
from typing import Any, List, Dict
from pathlib import Path

# Config

In [3]:
defaults = dict(
        gnn_model_type="CGNN-3D",
        split_type="random-k-fold",
        filter_rmsd_max_value=2,
        graph_level=True,
        split_index=1,
        batch_size=10,
        device="cpu",  # Default to CPU, can be changed to "cuda" for GPU
        dtype_out=None,  # None means no dtype conversion
    )

cfg.register("probing_ds", **defaults)
    # I might need to set some of these with arguments later
    # # Initializing config, filling it up as we go
    # cfg.register("probing_ds",
    #              gnn_model_type="CGNN-3D",
    #              split_type="random-k-fold",
    #              filter_rmsd_max_value=2,
    #              graph_level=True,
    #              split_index=0,
    #              dtype_out=None,  # None means no dtype conversion
    #              ) 

prob_config = cfg.get("probing_ds")


# Get the addresses; each follows a pattern
#   - model config and model ckpt follow: root/models/rmsd_cutoff_<rmsd_threshold>/<split_type>/<fold>/<model_name>
#   - splits follows: root/data/processed/filter_predicted_rmsd_le<rmsd_threshold>.00/<split_type>/<fold>_5.csv
#   - output_dir: root/data/probing/<model_name>/<split_type>/<fold>/output
model_dir = get_model_dir(rmsd_threshold = prob_config.filter_rmsd_max_value,
                                split_type = prob_config.split_type,
                                split_fold = prob_config.split_index,
                                model_type = prob_config.gnn_model_type
                                )
model_ckpt = get_model_ckpt(model_dir)
split_file_path = get_split_file(prob_config.split_type,
                                prob_config.split_index,
                                prob_config.filter_rmsd_max_value)
output_dir = get_out_dir(prob_config.gnn_model_type,
                            prob_config.split_type,
                            prob_config.split_index)

# Update the config from config file for GNN settings
# prob_config.update_from_file(get_gnn_config_path(model_dir))  This doesn't work for json, only for yaml
model_config = load_config(get_gnn_config_path(model_dir))
prob_config.update(
    {  **model_config,
        'model_ckpt': model_ckpt,
        'split_file': split_file_path,
        'output_dir': output_dir
    }, allow_duplicates=True
)

Config(gnn_model_type=CGNN-3D, split_type=random-k-fold, filter_rmsd_max_value=2, graph_level=True, split_index=1, batch_size=42, device=cpu, dtype_out=None, lr=0.0001, act=silu, ln1=True, ln2=True, ln3=True, seed=420, optim=adamw, epochs=300, k_fold=5, min_lr=1e-06, dropout=0, dry_run=False, loss_type=mse, lr_factor=0.9, num_heads=4, use_bonds=True, data_split=None, edge_types=[['ligand', 'intraacts', 'ligand'], ['ligand', 'interacts', 'pocket'], ['pocket', 'interacts', 'ligand']], graph_norm=False, node_types=['complex'], accelerator=gpu, lr_patience=8, num_workers=32, weight_decay=3e-06, edge_attr_size=4, need_distances=False, clip_grad_value=None, hidden_channels=256, remove_hydrogen=True, interaction_modes=['covalent', 'structural'], max_num_neighbors=16, add_docking_scores=False, interaction_radius=6, num_attention_blocks=3, num_residue_features=6, add_artificial_decoys=False, accumulate_grad_batches=3, early_stopping_patience=24, additional_atom_features=False, perturb_ligand_po

# gnn

In [4]:
gnn_model = build_gnn_model(prob_config).eval()
assert gnn_model is not None, "Failed to build GNN model"
gnn_model.to("cpu")
gnn_model

ComplexTransformer(
  (criterion): MSELoss()
  (act): SiLU()
  (interaction_module): CombinedInteractions(
    (interactions): ModuleList(
      (0): CovalentInteractions(
        (act): SiLU()
        (lin): Linear(in_features=4, out_features=256, bias=True)
      )
      (1): StructuralInteractions(
        (act): SiLU()
        (distance_embedding): GaussianDistEmbedding()
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
    )
    (act): SiLU()
  )
  (atomic_num_embedding): Embedding(100, 256)
  (lin_atom_features): Linear(in_features=12, out_features=256, bias=True)
  (attention_blocks): ModuleList(
    (0-2): 3 x SPAB(
      (attention): SparseAttention(
        (lin_query): Linear(in_features=256, out_features=256, bias=False)
        (lin_key_value): Linear(in_features=256, out_features=512, bias=False)
        (lin_bias): Linear(in_features=256, out_features=512, bias=False)
        (lin_out): Linear(in_features=256, out_features=256, bias=False)
   

# ds

Trying to see if I can throw away the slice of data that I do not use

In [5]:
from kinodata.data import KinodataDocked
from kinodata.data.data_split import Split
from kinodata.transform import TransformToComplexGraph
import gc

split_path = prob_config.split_file

if split_path is None:
    raise ValueError("split_path must be provided")
if isinstance(split_path, str):
    split_path = Path(split_path)
if split_path.exists():
    split = Split.from_csv(split_path)
else:
    raise FileNotFoundError(f"Split file not found: {split_path}")

full_ds = KinodataDocked(transform=TransformToComplexGraph(remove_heterogeneous_representation=False),
                    use_multiprocessing=True,
                    num_processes= os.cpu_count())
ds = full_ds[[*split.test_split, *split.val_split]]
del full_ds
gc.collect()

52

In [6]:
ds

KinodataDocked(8248)

In [5]:
ds = build_kd_ds(split_path=prob_config.split_file)
ds

KinodataDocked(8248)

In [7]:
print(prob_config.output_dir)
print(prob_config.device)

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/1
cpu


In [7]:
loader = DataLoader(ds, batch_size=prob_config.batch_size, shuffle=False)

# Pipeline

In [6]:
layer_bufs: Dict[str, List[torch.Tensor]] = {}
prior_buf: List[torch.Tensor] = []
idents: List[int] = []

max_batches = 1  # stop early for smoke/debug runs

In [8]:
with torch.no_grad():
    for i, batch in enumerate(tqdm(loader, desc="Computing graph representations")):
        if i >= max_batches:
            break
        # IDs
        batch_idents = batch.ident.tolist()
        idents.extend(batch_idents)

        # Move to model device
        batch = batch.to(gnn_model.device)

        if not prob_config.graph_level:
            raise NotImplementedError("Only graph-level probing is implemented for now.")
            _ , intermediate_node_reprs, intermediate_edge_reprs, _ = gnn_model(batch)

        # Forward
        _ , intermediate_node_reprs, _ , prior_readout = gnn_model(batch)
        
        # Pool each layer with the model's aggregator
        # intermediate_node_reprs: {layer_name: (node_repr, batch_index)}
        for layer_name, (node_repr, batch_index) in intermediate_node_reprs.items():
            graph_repr = gnn_model.aggr(node_repr, batch_index).detach().cpu()

            # Possible memory optimization
            if prob_config.dtype_out is not None:
                graph_repr = graph_repr.to(prob_config.dtype_out)

            # Create a buffer for each layer
            if layer_name not in layer_bufs:
                layer_bufs[layer_name] = []

            layer_bufs[layer_name].append(graph_repr)


Computing graph representations:   1%|          | 1/197 [00:13<42:56, 13.15s/it]


## Checking dims

In [9]:
ids_tensor = torch.tensor(idents, dtype=torch.long)
print(f"IDs tensor shape: {ids_tensor.shape}")
ids_tensor

IDs tensor shape: torch.Size([42])


tensor([ 34859,  12221,  15380,  63654,  70799,  39894,  25370, 104066,  27083,
         56640,  23290,   7083,  56635,   6690, 114702,  88841,  77157,  25611,
         41206,  44432,  15031,   3685,  86827, 110857,  95600,  83497,  55742,
        104487,  94733, 105919,  97832,  47257,  78389, 106790,  98106,   9511,
         76658,  75969, 103258,  76727,  47167,  86743])

In [10]:
for layer_name, chunks in layer_bufs.items():
    print(f"Layer {layer_name} has {len(chunks)} chunks of shape {chunks[0].shape}")

Layer layer_1 has 1 chunks of shape torch.Size([42, 256])
Layer layer_2 has 1 chunks of shape torch.Size([42, 256])
Layer layer_3 has 1 chunks of shape torch.Size([42, 256])


# cat

In [11]:
layers_cat: Dict[str, torch.Tensor] = {}
for layer_name, chunks in layer_bufs.items():
    layers_cat[layer_name] = torch.cat(chunks, dim=0)  # [N_fold, d]
print("Concatenated layers:")
for layer_name, tensor in layers_cat.items():
    print(f"{layer_name}: {tensor.shape}")

Concatenated layers:
layer_1: torch.Size([42, 256])
layer_2: torch.Size([42, 256])
layer_3: torch.Size([42, 256])


# check save

In [ ]:
def save_out_tensor(tensor: torch.Tensor, output_dir: Path, filename: str):
    """
    Save a tensor to a file in the output directory.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    out_file = output_dir / filename
    torch.save(tensor, out_file)
    return out_file

In [15]:
print(prob_config.output_dir)

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/1


In [ ]:
# Save to disk separately
for layer_name, layer_cat in layers_cat.items():
    save_out_tensor(layer_cat, prob_config.output_dir / str(prob_config.split_index), f"{layer_name}_{prob_config.split_index}.pt")
save_out_tensor(ids_tensor, prob_config.output_dir / str(prob_config.split_index), f"ids_{prob_config.split_index}.pt")

PosixPath('/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/1/ids_1.pt')

# check load

In [16]:
def load_out_tensor(output_dir: Path, filename: str) -> torch.Tensor:
    """
    Load a tensor from a file in the output directory.
    """
    out_file = output_dir / filename
    if not out_file.exists():
        raise FileNotFoundError(f"File {out_file} does not exist.")
    return torch.load(out_file)

In [ ]:
loaded_layers = {}
for layer_name in layers_cat.keys():
    loaded_layers[layer_name] = load_out_tensor(prob_config.output_dir / str(prob_config.split_index), f"{layer_name}_{prob_config.split_index}.pt")

In [33]:
assert len(loaded_layers) == len(layers_cat), "Not all layers were loaded correctly"
for layer_name, tensor in loaded_layers.items():
    assert tensor.shape == layers_cat[layer_name].shape, f"Shape mismatch for layer {layer_name}: {tensor.shape} vs {layers_cat[layer_name].shape}"

### check aggregate

In [4]:
from prob.prob_ds_helpers import aggregate_folds
from prob.generate_prob_ds import set_probing_config



# cause this is just checking:
prob_config.k_fold = 2

for layer_name in ["layer_1", "layer_2", "layer_3", "prior_readout"]:
    aggregate_folds(prob_config, layer_name)

Aggregated tensor saved to /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/layer_1.pt
Aggregated tensor saved to /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/layer_2.pt
Aggregated tensor saved to /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/random-k-fold/layer_3.pt
No tensors found for aggregation.


In [ ]:
loaded_tensors = {
    layer_name: load_out_tensor(prob_config.output_dir/str(prob_config.split_index), f"{layer_name}_{prob_config.split_index}.pt")
    for layer_name in ["layer_1", "layer_2", "layer_3"]
}

In [15]:
for layer_name, tensor in loaded_tensors.items():
    print(f"{layer_name}, Tensor shape: {tensor.shape}")

layer_1, Tensor shape: torch.Size([62, 256])
layer_2, Tensor shape: torch.Size([62, 256])
layer_3, Tensor shape: torch.Size([62, 256])
